## 데이터 수집 자동화

### 다나와 상품정보 스크래핑

#### 자동 로그인

- 패스워드 숨기기
1. setx 명령어로 시스템에 저장하는 방법
- powershell

```powershell
> setx DANAWA_ID "아이디"
```
```powershell
> setx DANAWA_PASSWORD "비밀번호"
```
→ 성공: 지정한 값을 저장했습니다.
- sysdm.cpl로 시스템 정보에서 환경변수 user 리스트에서 오픈되어 있음.

2. keyring 으로 윈도우 자격증명 저장소
- setx 와 달리 아예 오픈 안됨.
- 패키지 설치

In [1]:
!pip install keyring

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import keyring
from getpass import getpass

In [50]:
username = input('다나와 아이디')
password = getpass('다나와 비밀번호')

# 아이디 저장
keyring.set_password('danawa.com', 'user_id', username)
#패스워드
keyring.set_password('danawa.com', username, password)

print('아이디/패스워드 저장완료')

아이디/패스워드 저장완료


In [51]:
user_id = keyring.get_password('danawa.com', 'user_id')
user_id

'kcm0419'

In [53]:
password = keyring.get_password('danawa.com', user_id)
password is not None

True

- 패스워드 오픈

In [3]:
import pandas as pd
import os

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [135]:
driver = webdriver.Chrome()
wait = WebDriverWait(driver, 15)
driver.get('https://www.danawa.com')

# 보통 footer가 가장 아래이니 footer가 다 로딩될 때까지 ~
wait.until(
    EC.presence_of_element_located((By.ID, 'danawa_footer'))
)

<selenium.webdriver.remote.webelement.WebElement (session="b7e1d4824fb2e692c14718757832b527", element="f.5829A55C0672DD330219F983AE6BFB99.d.7E826F582F561EA025F1ACA603023CBC.e.4923")>

In [76]:
# 로그인 링크 확인 후 클릭
driver.find_element(By.LINK_TEXT, '로그인').click()

In [77]:
# 로그인하기


id_input = driver.find_element(By.CSS_SELECTOR, 'input[type="text"]')
id_input.send_keys(user_id)


pass_input = driver.find_element(By.CSS_SELECTOR, 'input[type="password"]')
pass_input.send_keys(password)

In [78]:
login_btn = driver.find_element(By.CLASS_NAME, 'btn_login')
login_btn.click()

#### 동적 페이지 스크래핑

- https://prod.danawa.com/list/?cate=11255834&15main_11_02
- 90개로 조정 후 시작

In [54]:
driver.get('https://prod.danawa.com/list/?cate=11255834&15main_11_02')
#wait.until(
#    EC.presence_of_element_located((By.TAG_NAME, 'footer'))
#)
wait.until(
    EC.visibility_of_element_located((By.TAG_NAME, 'footer'))
)

<selenium.webdriver.remote.webelement.WebElement (session="9d25cdec05e352957b1488a39abbb531", element="f.866C470DF510B97C21E8CC2BB56E4EAF.d.306A18A8102C09C2037EB71C49898D8D.e.149495")>

- select 태그 페이지 내에 1개만 존재

In [55]:
options = driver.find_elements(By.TAG_NAME, 'option')

#len(options)

for option in options:
    if option.get_attribute('value') == '90':
        option.click()

wait.until(
    EC.visibility_of_element_located((By.TAG_NAME, 'footer'))
)

<selenium.webdriver.remote.webelement.WebElement (session="9d25cdec05e352957b1488a39abbb531", element="f.866C470DF510B97C21E8CC2BB56E4EAF.d.306A18A8102C09C2037EB71C49898D8D.e.149495")>

In [7]:
products = driver.find_elements(By.CSS_SELECTOR, 'div[data-testid=ProductListItem]')
len(products)

90

In [8]:
product = products[0]
product.text

'1\n영웅컴퓨터 영웅 울트라게이밍 9800XS R7 9800X3D RTX5080\n9800X3D / RTX 5080 / (AMD) B650 / OS미포함 / 900W / AMD / 라이젠 9000시리즈 / 라이젠7 / 그래니트 릿지 / DDR5 / 32GB / M.2 / 1TB / NVIDIA / 그래픽 메모리: 16GB / 1Gbps 유선 / HDMI / DP포트 / 파워서플라이 / 미들타워 / 어항형 / 수랭쿨러 / LED쿨러 / 용도: 게임용\n혜택 최저가\n4,379,770원\n[G마켓] 삼성카드\n/\n무이자 최대 24개월\n기획전\n오늘 사는 PC가 가장 저렴한 PC!\n24.12. 등록\n브랜드로그\n4.8\n리뷰수\n(11)\n관심\n대량구매\n32GB, M.2 1TB\n4,529,810\n원\n5몰\nVS상품비교에 추가\nNew 새로워진 상품비교 기능을 이용해보세요.'

In [9]:
## 1번 인덱스 값에서 제품명, 제품스펙, 2번 인덱스 값에서 가격
product.find_elements(By.CSS_SELECTOR, '.dnw-product-list-item > div')

[<selenium.webdriver.remote.webelement.WebElement (session="9d25cdec05e352957b1488a39abbb531", element="f.866C470DF510B97C21E8CC2BB56E4EAF.d.5DFD30F5EEA9698832A838A1BAF05979.e.19465")>,
 <selenium.webdriver.remote.webelement.WebElement (session="9d25cdec05e352957b1488a39abbb531", element="f.866C470DF510B97C21E8CC2BB56E4EAF.d.5DFD30F5EEA9698832A838A1BAF05979.e.19466")>,
 <selenium.webdriver.remote.webelement.WebElement (session="9d25cdec05e352957b1488a39abbb531", element="f.866C470DF510B97C21E8CC2BB56E4EAF.d.5DFD30F5EEA9698832A838A1BAF05979.e.19467")>]

In [10]:
computer_info = product.find_elements(By.CSS_SELECTOR, '.dnw-product-list-item > div')[1]
computer_info.text

'1\n영웅컴퓨터 영웅 울트라게이밍 9800XS R7 9800X3D RTX5080\n9800X3D / RTX 5080 / (AMD) B650 / OS미포함 / 900W / AMD / 라이젠 9000시리즈 / 라이젠7 / 그래니트 릿지 / DDR5 / 32GB / M.2 / 1TB / NVIDIA / 그래픽 메모리: 16GB / 1Gbps 유선 / HDMI / DP포트 / 파워서플라이 / 미들타워 / 어항형 / 수랭쿨러 / LED쿨러 / 용도: 게임용\n혜택 최저가\n4,379,770원\n[G마켓] 삼성카드\n/\n무이자 최대 24개월\n기획전\n오늘 사는 PC가 가장 저렴한 PC!\n24.12. 등록\n브랜드로그\n4.8\n리뷰수\n(11)\n관심\n대량구매'

In [11]:
# 컴퓨터 이름 추출
computer_info.find_element(By.CSS_SELECTOR, 'div a').text

'영웅컴퓨터 영웅 울트라게이밍 9800XS R7 9800X3D RTX5080'

In [12]:
# 컴퓨터 사양
specs = computer_info.find_elements(By.CSS_SELECTOR, 'div[data-testid="ProductListSpecs"]')
' / '.join(spec.text for spec in specs)

'9800X3D / RTX 5080 / (AMD) B650 / OS미포함 / 900W / AMD / 라이젠 9000시리즈 / 라이젠7 / 그래니트 릿지 / DDR5 / 32GB / M.2 / 1TB / NVIDIA / 그래픽 메모리: 16GB / 1Gbps 유선 / HDMI / DP포트 / 파워서플라이 / 미들타워 / 어항형 / 수랭쿨러 / LED쿨러 / 용도: 게임용'

In [46]:
price_info = product.find_elements(By.CSS_SELECTOR, '.dnw-product-list-item > div')[2]
price_info.text

'32GB, M.2 1TB\n4,529,810\n원\n5몰\nVS상품비교에 추가\nNew 새로워진 상품비교 기능을 이용해보세요.'

In [47]:
# 인덱스로 데이터 가져오면서 필요값만 필터링
# 원이 나오면 반복 종료
price_info.find_elements(By.CSS_SELECTOR, 'li > div span')[1].text

'4,529,810'

In [49]:
spans = price_info.find_elements(By.CSS_SELECTOR, 'li > div span')

for span in spans:
    if('위' in span.text): continue
    if('원' in span.text): break
    print(span.text)

32GB, M.2 1TB
4,529,810


- 각 부분별 동작확인 완료

#### 한 페이지 90개 데이터 스크래핑

In [74]:
# 다나와 홈페이지 게이밍 pc 페이지 이동
#driver.get('https://prod.danawa.com/list/?cate=11255834&15main_11_02')
#wait.until(
#    EC.visibility_of_element_located((By.TAG_NAME, 'footer'))
#)
# 게시물 개수 90개로
#options = driver.find_elements(By.TAG_NAME, 'option')

#len(options)

#for option in options:
#    if option.get_attribute('value') == '90':
#        option.click()

#wait.until(
#    EC.visibility_of_element_located((By.TAG_NAME, 'footer'))
#)

#총게시물 읽어오기
#products = driver.find_elements(By.CSS_SELECTOR, 'div[data-testid=ProductListItem]')
#len(products)

<selenium.webdriver.remote.webelement.WebElement (session="9d25cdec05e352957b1488a39abbb531", element="f.866C470DF510B97C21E8CC2BB56E4EAF.d.B5B261C16D9FBEE2060A9C4F1E19AEAF.e.533141")>

In [136]:
# 다나와 홈페이지 게이밍 pc 페이지 이동
driver.get('https://prod.danawa.com/list/?cate=11255834&15main_11_02')
wait.until(
    # 모든 요소가 화면에 완전 표시될 때까지 대기
    EC.visibility_of_all_elements_located((By.TAG_NAME, 'footer'))
)
# 게시물 개수 90개로
options = driver.find_elements(By.TAG_NAME, 'option')

#len(options)

for option in options:
    if option.get_attribute('value') == '90':
        option.click()

# 화면에 90개 데이터를 다 조회할 때까지 대기
products = wait.until(
    EC.presence_of_all_elements_located((By.CSS_SELECTOR, 'div[data-testid=ProductListItem]'))
)

danawa_prices = []

#총게시물 읽어오기
#len(products)

for index, product in enumerate(products):
   # print(index, product.text)
   items = product.find_elements(By.CSS_SELECTOR, '.dnw-product-list-item > div')
   computer_info = items[1] #컴퓨터명, 사양 정보 들어있는 영역
   price_info = items[2] #가격정보 들어있는 영역

   #print(price_info.text)
   ## Pandas DF에 컬럼으로 사용할 이름들 사용
   computer_name = computer_info.find_element(By.CSS_SELECTOR, 'div a').text
   specs = computer_info.find_elements(By.CSS_SELECTOR, 'div[data-testid="ProductListSpecs"]')
   computer_spec = ' / '.join(spec.text for spec in specs)

   # 가격
   spans = price_info.find_elements(By.CSS_SELECTOR, 'li > div span')
   price_spec = ''
   price = ''

   
   for span in spans:
    if('위' in span.text): continue # 순위 제외
    if('원' in span.text): break # 첫번째 금액만 받고 종료

    if('B' in span.text):
       price_spec = span.text
    else:
       price = span.text

   danawa_prices.append({
    'index' : index,
    'computer_name' : computer_name,
    'computer_spec' : computer_spec,
    'price_spec' : price_spec,
    'price' : price
   })

In [137]:
len(danawa_prices)

90

In [138]:
## Pandas DF로 변환
df_danawa = pd.DataFrame(danawa_prices)

In [140]:
df_danawa.head()

,index,computer_name,computer_spec,price_spec,price
0,0,영웅컴퓨터 영웅 울트라게이밍 9800XS R7 9800X3D RTX5080,9800X3D / RTX 5080 / (AMD) B650 / OS미포함 / 900W...,"32GB, M.2 1TB","4,523,480"
1,1,영웅컴퓨터 영웅 875 게이밍울트라560X R5 5600 RTX5060,5600 / RTX 5060 / (AMD) B550 / OS미포함 / 600W / ...,"16GB, M.2 256GB","1,181,800"
2,2,포유컴퓨터 퍼포먼스PC 47 R5 7500F RTX5060Ti,7500F / RTX 5060 Ti / (AMD) B650 / OS미포함 / 700...,"16GB, M.2 500GB","1,579,200"
3,3,MSI MAG Codex Z2 A9NVR A18 R7-9800X3D RTX 5070...,9800X3D / RTX 5070 Ti / (AMD) B850 / OS미포함 / 8...,"32GB, M.2 1TB","3,943,540"
4,4,영웅컴퓨터 영웅 게이밍울트라 7857 R7 7800X3D RTX5070,7800X3D / RTX 5070 / (AMD) B650 / OS미포함 / 800W...,"32GB, M.2 1TB","2,805,860"


##### 페이지 확인

In [141]:
#
buttons = driver.find_elements(By.CSS_SELECTOR, 'nav > nav div button')
len(buttons)

10

In [142]:
for button in buttons:
    print(button.text)

1
2
3
4
5
6
7
8
9
10


In [143]:
button_nextpage = buttons[9]
button_nextpage.click()

In [144]:
# 다 동작했을 때 10페이지에 있는지 확인
for button in buttons:
    button.click()

    products = wait.until(
        EC.presence_of_all_elements_located((By.CSS_SELECTOR, 'div[data-testid=ProductListItem]'))
    )

In [151]:
driver = webdriver.Chrome()
wait = WebDriverWait(driver, 15)
driver.get('https://www.danawa.com')

# 보통 footer가 가장 아래이니 footer가 다 로딩될 때까지 ~
wait.until(
    EC.presence_of_element_located((By.ID, 'danawa_footer'))
)

<selenium.webdriver.remote.webelement.WebElement (session="4277e41c6cfd5fa00e2854f47bfc5c12", element="f.43808C818392FFAAEECDDE284744FC79.d.92F9752A90C705D9C333F8307A096A01.e.3349")>

##### 이전 스크래핑 + 페이지 이동 조합

In [154]:
# 다나와 홈페이지 게이밍 pc 페이지 이동
driver.get('https://prod.danawa.com/list/?cate=11255834&15main_11_02')
wait.until(
    # 모든 요소가 화면에 완전 표시될 때까지 대기
    EC.visibility_of_all_elements_located((By.TAG_NAME, 'footer'))
)
# 게시물 개수 90개로
options = driver.find_elements(By.TAG_NAME, 'option')

#len(options)

for option in options:
    if option.get_attribute('value') == '90':
        option.click()

danawa_prices = []
buttons = driver.find_elements(By.CSS_SELECTOR, 'nav > nav div button')

# 다 동작했을 때 10페이지에 있는지 확인
for button in buttons:
    button.click()
    page_num = int(button.text)

    products = wait.until(
        EC.presence_of_all_elements_located((By.CSS_SELECTOR, 'div[data-testid=ProductListItem]'))
    )

    # 한페이지 스크래핑 내용 여기
    #총게시물 읽어오기
#len(products)

    for index, product in enumerate(products):
    # print(index, product.text)
        items = product.find_elements(By.CSS_SELECTOR, '.dnw-product-list-item > div')
        computer_info = items[1] #컴퓨터명, 사양 정보 들어있는 영역
        price_info = items[2] #가격정보 들어있는 영역

    #print(price_info.text)
    ## Pandas DF에 컬럼으로 사용할 이름들 사용
        computer_name = computer_info.find_element(By.CSS_SELECTOR, 'div a').text
        specs = computer_info.find_elements(By.CSS_SELECTOR, 'div[data-testid="ProductListSpecs"]')
        computer_spec = ' / '.join(spec.text for spec in specs)

    # 가격
        spans = price_info.find_elements(By.CSS_SELECTOR, 'li > div span')
        price_spec = ''
        price = ''

    
        for span in spans:
            if('위' in span.text): continue # 순위 제외
            if('원' in span.text): break # 첫번째 금액만 받고 종료

            if('B' in span.text):
                price_spec = span.text
            else:
                price = span.text

        danawa_prices.append({
            'index' : index,
            'computer_name' : computer_name,
            'computer_spec' : computer_spec,
            'price_spec' : price_spec,
            'price' : price
        })

In [155]:
df_total = pd.DataFrame(danawa_prices)
df_total.info()

<class 'pandas.DataFrame'>
RangeIndex: 900 entries, 0 to 899
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   index          900 non-null    int64
 1   computer_name  900 non-null    str  
 2   computer_spec  900 non-null    str  
 3   price_spec     900 non-null    str  
 4   price          900 non-null    str  
dtypes: int64(1), str(4)
memory usage: 35.3 KB


In [158]:
# 가격 비교예정으로 금액 없음
df_total[df_total['price'].isna()]

,index,computer_name,computer_spec,price_spec,price


In [ ]:
df_total = df_total.dropna()

In [150]:
# 판다스 컬럼에 천단위 쉼표(,)는 제거후 형변환
df_total['price'] = df_total['price'].str.replace(',','')

In [ ]:
df_total['price'] = pd.to_numeric